# [실습] LLM API, Structured Output & Tool Calling

LangChain을 활용하여 다양한 LLM API를 호출하고, Tool Calling과 Structured Output을 실습합니다.

## 학습 목표

- `init_chat_model`로 LLM(GPT-5.2) 초기화 방법 학습
- Structured Output (Pydantic 바인딩)으로 구조화된 데이터 생성
- `@tool` 데코레이터로 커스텀 도구를 정의하고 사용하기
- Tool Calling 메커니즘: `tool_calls` 메시지 구조 이해


## 환경 설정

In [2]:
%pip install langchain langchain-openai langchain-core -q

Note: you may need to restart the kernel to use updated packages.


In [6]:
from dotenv import load_dotenv
import os

load_dotenv('.env', override=True)

if os.environ.get('OPENAI_API_KEY'):
    print('OpenAI API 키 설정 완료!')
else:
    print('OpenAI API 키가 설정되지 않았습니다. .env 파일을 확인하세요.')

OpenAI API 키 설정 완료!


---
## 1. LLM 초기화하기

LangChain 1.x에서는 `init_chat_model`을 사용하여 다양한 LLM을 통일된 인터페이스로 초기화할 수 있습니다.

### 1-1. init_chat_model (권장 방식)

`init_chat_model`은 모델 이름으로부터 자동으로 Provider를 감지합니다.

In [9]:
from langchain.chat_models import init_chat_model

# GPT-5.2 초기화 (OpenAI)
gpt = init_chat_model('gpt-5.2', max_tokens=16384,
    # timeout = 60, # 60초 이상 응답이 없으면 종료
    # max_retries = 6, # 실패 시 재시도 횟수 (기본값 6, 비활성화: 1)
    )
print(f'LLM 모델 초기화 완료!')
print(f'모델 이름: {gpt.model_name}')

LLM 모델 초기화 완료!
모델 이름: gpt-5.2


### 1-2. Provider별 클래스 지정

특정 Provider의 모델을 사용하는 경우, 클래스를 직접 지정하기도 합니다.   
- `ChatOpenAI, ChatGoogleGenerativeAI, ChatOllama, ChatHuggingFace` 등

In [10]:
from langchain_openai import ChatOpenAI

gpt_direct = ChatOpenAI(model='gpt-5.2', max_tokens=16384)
gpt_direct

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-openai': '1.3.3'}}, output_version=None, profile={'name': 'GPT-5.2', 'release_date': '2025-12-11', 'last_updated': '2025-12-11', 'open_weights': False, 'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001FEDE3C3D90>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001FEDE3C81D0>, root_client=<openai.OpenAI object at 

### 1-3. LLM에 메시지 전달하기   

LangChain의 모든 모듈은 `invoke()`를 통해 실행합니다.   
LLM을 invoke하면 `AIMessage` 클래스를 return합니다.

In [11]:
# 1. 문자열을 통한 입력
response = gpt.invoke('안녕하세요! 한 문장으로 자기 소개 부탁드립니다.')
print(f'GPT 응답: {response.text}')
response

GPT 응답: 저는 질문에 답하고 글쓰기·요약·번역·아이디어 정리 등을 도와드리는 AI 어시스턴트입니다.


AIMessage(content='저는 질문에 답하고 글쓰기·요약·번역·아이디어 정리 등을 도와드리는 AI 어시스턴트입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 18, 'total_tokens': 53, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.2-2025-12-11', 'system_fingerprint': None, 'id': 'chatcmpl-Du6fQKxXaBozm7PI2qNET561p1gEL', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ef73c-a648-7151-93ab-cdd9e99f49f7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 35, 'total_tokens': 53, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

`stream`은 토큰이나 이벤트 단위로 결과물을 출력합니다.

In [13]:
for chunk in gpt.stream('AI Agent의 핵심 구성 요소를 세 단어로 표현해보세요.'):
    print(chunk.text, end='', flush=True)

- **인지(Perception)**  
- **추론(Reasoning/Planning)**  
- **행동(Action/Execution)**

위에서 입력한 문자열은 내부적으로 `HumanMessage` 클래스로 LLM에 전달됩니다.    

보다 자세한 입력을 위해, 랭체인의 내부 클래스를 직접 활용합니다.

### 1-4. Message List를 통한 입력

In [15]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

# System Message: 첫 번째로 들어가는 지시사항이나 지침, Human Message보다 우선순위가 높음

msg_list = [
    SystemMessage('답변을 마친 후에는, 질문 형식으로 다음 작업에 대해 제안하세요.'),
    HumanMessage('Agent와 LLM의 차이에 대해 한 문장으로 설명해주세요.')
    # AIMessage
    # HumanMessage
]

stream = gpt.stream(msg_list)

for s in stream:
    print(s.text, end='', flush=True)

LLM은 텍스트를 이해·생성하는 언어 모델 자체이고, Agent는 LLM을 포함해 도구 사용·기억·계획·실행을 결합하여 목표를 스스로 수행하는 시스템입니다.

Agent가 실제로 어떤 도구(예: 검색, 코드 실행, 일정 관리)를 사용해 작업을 수행하는지 예시로도 한 문장으로 들어드릴까요?

---
## 2. Prompt Template과 Chain   

프롬프트 템플릿을 사용하면 입력 변수를 포함한 재사용 가능한 프롬프트를 작성할 수 있습니다.

`ChatPromptTemplate`은 시스템 프롬프트와 사용자 프롬프트를 구분하여 설정합니다.

In [17]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate([
    ('system', '''2025-2026년까지의 최신 지식을 최대한 활용하여 답변하세요. 
간결하게 3문장 이내로 답변하세요. 이모지를 많이 사용하세요. 최소 10개'''),
    ('human', '{question}') # Field
])

# 프롬프트와 LLM을 | (파이프)로 연결하면 체인(Chain)이 됩니다
chain = prompt | gpt

response = chain.invoke('LLM의 최근 컨텍스트 윈도우는 어디까지 확장되고 있나요?')

print(response.text)

2025~2026년 기준, 상용 LLM의 컨텍스트 윈도우는 보편적으로 **128k 토큰**이 널리 쓰이고, 일부는 **최대 ~200k+**까지 제공되며, 연구·실험/특정 제품군에서는 **~1M 토큰급(롱컨텍스트)**도 보고·제공되는 흐름입니다. 📈🧠🪟🔍📚🚀⚙️🧩📝🔥  
다만 길이가 늘수록 **비용·지연·정확도(중간/후반부 회수 저하 등)** 문제가 커져, 실제 운영에서는 **RAG/요약/메모리 전략**과 병행하는 경우가 많습니다. 💸⏱️🎯📉🧷🗂️🧠🔁🛠️✨  
정확한 “어디까지”는 **모델/버전/요금제**에 따라 다르니, 쓰는 제품을 알려주면 해당 최신 스펙 기준으로 정리해드릴게요. 🏷️📌🔎📊✅🤝😊🎉🧠🪟


입력 변수가 두 개 이상일 때는 dict 형태로 전달합니다.

In [18]:
analysis_prompt = ChatPromptTemplate([
    ('system', '''당신은 LLM Agent 설계 전문가입니다.
주어진 문제를 해결하는 Agent의 Tool 목록과 주요 Failure Scenario를 각각 {num}개 이내의 Bullet으로 설명하세요.'''),
    ('human', '문제: {problem}')
])

chain = analysis_prompt | gpt

# 단일 호출
response = chain.invoke({'problem': '사내 이메일 자동화 에이전트', 'num': '3'})
print(response.text)

### Tool 목록 (3개 이내)
- **Email API Tool (Gmail/Outlook/Exchange 커넥터)**  
  메일 조회/검색, 스레드 추적, 라벨·폴더 이동, 발송(답장/전달), 초안 생성, 첨부파일 업·다운로드.
- **사내 시스템 연동 Tool (HR/ERP/CRM/헬프데스크/캘린더)**  
  인입 메일 내용을 기반으로 티켓 생성/상태조회, 고객·직원 정보 조회, 일정 생성/회의실 예약, 결재/요청서 생성 등 업무 트리거.
- **Policy & Safety Tool (규정/권한/RAG + DLP/PII 마스킹 + 감사로그)**  
  사내 규정(보안/대외 커뮤니케이션/결재선) 검색, 권한 기반 실행(승인 워크플로), 민감정보 탐지·마스킹, 모든 액션에 대한 감사 추적.

---

### 주요 Failure Scenario (3개 이내)
- **오발송/수신자 오류 및 스레드 혼선**  
  유사한 이름/그룹메일 자동완성, 외부 도메인 포함, 잘못된 “전체답장”, 다른 스레드에 답장해 정보가 섞이는 문제.
- **정보보안 사고(기밀/PII/내부자료 유출)**  
  첨부파일·본문의 민감정보를 외부로 전달, 권한 없는 사용자 대신 정보 조회/전송, 규정 위반 문구(계약/법무) 자동 발송.
- **업무 자동화의 잘못된 실행(티켓/결재/일정 오처리)**  
  메일 의도 오해로 잘못된 티켓 생성·우선순위 설정, 중복 처리(같은 요청 반복 생성), 캘린더/헬프데스크 상태를 잘못 업데이트하여 운영 혼선 발생.



또한 `batch()`를 사용하면 여러 입력을 한 번에 처리할 수 있습니다.

In [19]:
# batch 호출: 여러 입력을 한 번에 처리
problems = [
    {'problem': '논문 요약 에이전트', 'num': '3'},
    {'problem': '코드 리뷰 에이전트', 'num': '3'},
]
results = chain.batch(problems)

for i, result in enumerate(results):
    print(f'--- {problems[i]["problem"]} ---')
    print(result.text)
    print()

--- 논문 요약 에이전트 ---
### Tool 목록 (3개 이내)
- **PDF/문서 수집·파싱 도구**: URL/DOI/arXiv로 논문을 가져오고, PDF를 텍스트/섹션(초록, 서론, 방법, 실험, 결론) 단위로 추출(OCR 포함)하는 도구.
- **구조화 요약·출력 포맷터**: 섹션별 핵심 주장/기여/방법/결과/한계를 템플릿(JSON/Markdown)으로 정리하고, 길이(한 문단/1페이지), 대상(비전공/전공), 언어(한/영) 옵션을 적용하는 도구.
- **근거 추적(인용/하이라이트) 도구**: 요약 문장마다 근거가 되는 원문 위치(페이지/문단/문장)와 인용구를 매핑해 “요약-근거” 연결을 제공하는 도구.

### 주요 Failure Scenario (3개 이내)
- **파싱 실패/누락으로 인한 왜곡**: 수식·표·그림 캡션·각주가 추출되지 않거나 섹션 경계가 깨져 핵심 결과(예: 성능 수치, 조건)가 누락/변형됨.
- **과도한 압축으로 인한 핵심 맥락 손실**: 문제 설정/가정/데이터셋/평가지표를 빼고 결론만 요약해, 논문의 실제 기여 대비 과장되거나 오해를 유발.
- **근거 없는 추론(환각) 및 잘못된 인용 연결**: 논문에 없는 실험/비교/한계점을 만들어내거나, 요약 문장에 잘못된 페이지/문장을 근거로 붙여 신뢰성을 훼손.

--- 코드 리뷰 에이전트 ---
## Tool 목록 (3개 이내)

- **Repo/Code Access Tool (Git/Filesystem/PR API)**  
  PR diff, 전체 파일 컨텍스트, 히스토리(Blame/commit), 관련 이슈/티켓을 읽어와 “변경 의도 + 영향 범위”를 파악하는 데 사용.

- **Static Analysis & Quality Tool (Linter/SAST/Typecheck/Test Runner)**  
  언어별 lint, 타입체크, 보안 정적분석(SAST), 유닛/통합 테스트 실행 결과를 수집해 리뷰 코멘트를 “재현 가능한 근거”로 뒷받침.

- **Policy/Guide

---
## 3. Structured Output (구조화된 출력)

LLM의 결과물을 모듈화하고 활용하기 위해서는 보통 출력 텍스트의 후처리가 필요합니다.   

LLM의 출력을 구조화하는 방법에 대해 알아보겠습니다.   

Pydantic은 파이썬을 위한 데이터 클래스로, 

`with_structured_output()`을 사용하면 LLM이 지정된 스키마에 맞는 JSON을 출력하고, 자동으로 Pydantic 객체로 변환됩니다.

### 3-1. 기본 Structured Output

In [24]:
from pydantic import BaseModel, Field
from typing import List

class MovieReview(BaseModel):
    """영화 리뷰 분석 결과"""
    title: str = Field(description='영화 제목')
    keywords: List[str] = Field(description='핵심 키워드 3개')
    summary: str = Field(description='한 문장 요약')
    sentiment: str = Field(description='감정 분석 결과 (긍정/부정/중립)')
    score: float = Field(description='평점 (별 1개~5개)')

# with_structured_output으로 Pydantic 모델 바인딩
structured_llm = gpt.with_structured_output(MovieReview)
# 출력 형식에 대한 시스템 프롬프트 삽입 + 자동 파싱

review = structured_llm.invoke(
    '매드 맥스: 분노의 도로는 액션 영화의 새로운 기준을 세운 작품입니다. 조지 밀러 감독이 '
    '70대에 만들었다는 사실이 믿기지 않을 만큼 에너지가 넘쳤어요. 황량한 사막을 가로지르는 '
    '추격전이 영화의 거의 전부를 차지하지만, CG에 의존하지 않고 실제 차량과 스턴트로 '
    '찍어낸 장면들은 압도적인 현장감을 줍니다. 대사를 최소화하고 이미지와 움직임만으로 '
    '서사를 끌고 가는 연출이 인상적이었고, 퓨리오사라는 강렬한 캐릭터를 통해 단순한 '
    '액션을 넘어선 메시지까지 담아냈습니다. 다만 쉴 틈 없이 몰아치는 전개 탓에 인물의 '
    '내면을 들여다볼 여유가 부족했던 점은 아쉬웠습니다. 그럼에도 극장에서 다시 보고 싶은, '
    '체험에 가까운 영화였습니다.'
)
print(f'타입: {type(review).__name__}')
print(f'제목: {review.title}')
print(f'키워드: {review.keywords}')
print(f'요약: {review.summary}')
print(f'감정: {review.sentiment}')
print(f'평점: {review.score}')

review

타입: MovieReview
제목: 매드 맥스: 분노의 도로
키워드: ['실제 스턴트', '추격전', '퓨리오사']
요약: CG보다 실물 액션과 이미지 중심 연출로 압도적 질주를 선사하면서도 퓨리오사를 통해 메시지까지 담아낸 체험형 액션 영화다.
감정: 긍정
평점: 4.5


MovieReview(title='매드 맥스: 분노의 도로', keywords=['실제 스턴트', '추격전', '퓨리오사'], summary='CG보다 실물 액션과 이미지 중심 연출로 압도적 질주를 선사하면서도 퓨리오사를 통해 메시지까지 담아낸 체험형 액션 영화다.', sentiment='긍정', score=4.5)

### 3-2. 중첩된 Structured Output

Pydantic 클래스를 중첩하면 복잡한 구조의 데이터도 생성할 수 있습니다.

In [26]:
class DailySchedule(BaseModel):
    """하루 일정"""
    morning: str = Field(description='오전 일정')
    afternoon: str = Field(description='오후 일정')
    evening: str = Field(description='저녁 일정')

class TravelPlan(BaseModel):
    """여행 계획"""
    destination: str = Field(description='목적지')
    duration: str = Field(description='여행 기간')
    schedule: List[DailySchedule] = Field(description='일별 일정')

In [27]:
trip_planner = gpt.with_structured_output(TravelPlan)

trip_prompt = ChatPromptTemplate([
    ('system', '당신은 여행 계획 전문가입니다. 주어진 조건에 맞는 여행 계획을 작성하세요.'),
    ('human', '{destination}에서 {duration} 여행 계획을 세워주세요.')
])

# 프롬프트 + Structured Output LLM 체인
trip_chain = trip_prompt | trip_planner
plan = trip_chain.invoke({'destination': '제주도', 'duration': '2일'})

print(f'목적지: {plan.destination}')
print(f'기간: {plan.duration}')
for i, day in enumerate(plan.schedule, 1):
    print(f'\n{i}일차:')
    print(f'  오전: {day.morning}')
    print(f'  오후: {day.afternoon}')
    print(f'  저녁: {day.evening}')

목적지: 제주도
기간: 2일

1일차:
  오전: [1일차] 제주공항 도착 → 렌터카 수령 → 애월 해안도로 드라이브(한담해변 산책)
  오후: 애월/협재 방면 이동 → 협재해수욕장·금능해변(물때 맞으면 해변 산책) → 한림공원 또는 오설록 티뮤지엄(취향 선택)
  저녁: 서귀포 이동 → 천지연폭포 야간 산책(운영시간 확인) → 서귀포 올레시장 야시장/먹거리 투어 → 숙소 체크인

2일차:
  오전: [2일차] 성산/동부로 이동 → 성산일출봉(이른 시간 추천) 또는 섭지코지 산책
  오후: 우도(시간 여유 시) 자전거/스쿠터로 한 바퀴 또는 비자림·사려니숲길 산책(비 오는 날 대안) → 제주 시내 복귀
  저녁: 용두암 또는 이호테우해변 일몰 감상 → 공항 근처 저녁식사(흑돼지/고기국수) → 렌터카 반납 및 출발


---
## 4. Tool Calling

LLM Agent의 가장 주요한 기능은 도구를 호출하는 Tool Calling입니다.   

이는 명령어와 같은 형식의 구조화된 텍스트를 출력해, 해당 도구의 실행을 자동화하는 방식입니다.  

이 Tool Calling은 Structured Output의 특별한 형태에 해당합니다. 

LangChain에서는 `@tool` 데코레이터를 사용하여 파이썬 함수를 LLM이 사용할 수 있는 도구로 변환합니다.

tool의 정보는 LLM에 프롬프트를 통해 전달되어, 필요한 순간에 LLM이 도구를 사용할 수 있게 합니다.   

**아래의 정보가 LLM의 시스템 프롬프트에 포함됩니다.**

- 도구의 이름
- 입력 변수 형식
- 설명(docstring)

In [29]:
from langchain.tools import tool
from datetime import datetime

@tool
def add(a: int, b: int) -> int:
    """두 정수를 더합니다.

    Args:
        a: 첫 번째 정수
        b: 두 번째 정수
    """
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """두 정수를 곱합니다.

    Args:
        a: 첫 번째 정수
        b: 두 번째 정수
    """
    return a * b

@tool
def get_current_date() -> str:
    """현재 날짜를 YYYY-MM-DD 형식으로 반환합니다."""
    return datetime.now().strftime('%Y-%m-%d')

# 도구 직접 실행 테스트
print(f"add(3, 5) = {add.invoke({'a': 3, 'b': 5})}")
print(f"multiply(4, 7) = {multiply.invoke({'a': 4, 'b': 7})}")
print(f"get_current_date() = {get_current_date.invoke({})}")

add(3, 5) = 8
multiply(4, 7) = 28
get_current_date() = 2026-06-24


### 4-1. 도구 정보 확인

LLM은 도구의 이름, 설명, 파라미터 스키마를 통해 어떤 도구를 언제 사용할지 결정합니다.

In [30]:
print(f'도구 이름: {add.name}')
print(f'도구 설명: {add.description}')
print(f'\n파라미터 스키마 (JSON Schema):')
add.args_schema.model_json_schema()

도구 이름: add
도구 설명: 두 정수를 더합니다.

    Args:
        a: 첫 번째 정수
        b: 두 번째 정수

파라미터 스키마 (JSON Schema):


{'description': '두 정수를 더합니다.\n\nArgs:\n    a: 첫 번째 정수\n    b: 두 번째 정수',
 'properties': {'a': {'title': 'A', 'type': 'integer'},
  'b': {'title': 'B', 'type': 'integer'}},
 'required': ['a', 'b'],
 'title': 'add',
 'type': 'object'}

---
## 5. Tool Calling 메커니즘



LLM에 도구를 연결하면, LLM은 사용자의 질문에 따라 적절한 도구를 선택하고 호출 인자를 생성합니다.

LLM의 역할은 도구를 실행하기 위한 구조화된 텍스트를 생성하는 것입니다.   
이는 `name`, `args`를 포함합니다.

### 5-1. LLM에 도구 연결 (bind_tools)

In [52]:
tools = [add, multiply, get_current_date]

# bind_tools: LLM에 도구 정보를 프롬프트에 포함
llm_with_tools = gpt.bind_tools(tools)
# with_structured_output과 유사
print(llm_with_tools)

bound=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-openai': '1.3.3'}}, output_version=None, profile={'name': 'GPT-5.2', 'release_date': '2025-12-11', 'last_updated': '2025-12-11', 'open_weights': False, 'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001FEDE3C2410>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001FEDE3C2D50>, root_client=<openai.OpenAI obje

### 5-2. Tool Call 메시지 분석

LLM이 도구 사용이 필요하다고 판단하면, `AIMessage`의 `tool_calls` 필드에 Tool Call 정보를 출력합니다.

In [53]:
from rich import print as rprint

messages = [HumanMessage('3과 5를 더해주세요.')]

response = llm_with_tools.invoke(messages)

rprint(response)

AIMessage(
    content='',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 20,
            'prompt_tokens': 214,
            'total_tokens': 234,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 0,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'gpt-5.2-2025-12-11',
        'system_fingerprint': None,
        'id': 'chatcmpl-Du81TCrrx8NAb28rGjjLuiL4HS2yL',
        'service_tier': 'default',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--019ef78c-2a9d-76e3-b44e-2b97e9340883-0',
    tool_calls=[
        {'name': 'add', 'args': {'a': 3, 'b': 5}, 'id': 'call_xsZpsnhFHacgpL0buCziVOI4', 'type': 'tool_call'}
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 214,
        'output_tokens': 20,
        'total_tokens': 234,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'audio': 0, 'reasoning': 0}
    }
)

In [54]:
# Tool Call 상세 분석
if response.tool_calls:
    tool_call = response.tool_calls[0]
    print('--- Tool Call 구조 ---')
    print(f'  name (도구 이름): {tool_call["name"]}')
    print(f'  args (호출 인자): {tool_call["args"]}')
    print(f'  id (호출 ID):     {tool_call["id"]}')
    print(f'  type:             {tool_call["type"]}')

--- Tool Call 구조 ---
  name (도구 이름): add
  args (호출 인자): {'a': 3, 'b': 5}
  id (호출 ID):     call_xsZpsnhFHacgpL0buCziVOI4
  type:             tool_call


### 5-3. 도구 실행 및 결과 전달

Tool Call을 받으면 해당 도구를 실행하고, 결과를 `ToolMessage`로 LLM에 다시 전달합니다.

In [55]:
# 도구 이름 -> 도구 함수 매핑
tools_map = {t.name: t for t in tools}
print(f'등록된 도구: {list(tools_map.keys())}')

rprint(tools_map)

등록된 도구: ['add', 'multiply', 'get_current_date']


{
    'add': StructuredTool(
        name='add',
        description='두 정수를 더합니다.\n\n    Args:\n        a: 첫 번째 정수\n        b: 두 번째 정수',
        args_schema=<class 'langchain_core.utils.pydantic.add'>,
        func=<function add at 0x000001FEDE4DA660>
    ),
    'multiply': StructuredTool(
        name='multiply',
        description='두 정수를 곱합니다.\n\n    Args:\n        a: 첫 번째 정수\n        b: 두 번째 정수',
        args_schema=<class 'langchain_core.utils.pydantic.multiply'>,
        func=<function multiply at 0x000001FEDE4D8CC0>
    ),
    'get_current_date': StructuredTool(
        name='get_current_date',
        description='현재 날짜를 YYYY-MM-DD 형식으로 반환합니다.',
        args_schema=<class 'langchain_core.utils.pydantic.get_current_date'>,
        func=<function get_current_date at 0x000001FEDE4DA020>
    )
}

In [56]:

# Tool Call의 결과를 ToolMessage로 변환
tool_result = tools_map[tool_call['name']].invoke(tool_call)
#                    add                  .invoke
print(f'\nToolMessage: {tool_result}')
print(f'결과값: {tool_result.text}')
print(f'tool_call_id: {tool_result.tool_call_id}')

rprint(tool_result)


ToolMessage: content='8' name='add' tool_call_id='call_xsZpsnhFHacgpL0buCziVOI4'
결과값: 8
tool_call_id: call_xsZpsnhFHacgpL0buCziVOI4


ToolMessage(content='8', name='add', tool_call_id='call_xsZpsnhFHacgpL0buCziVOI4')

In [57]:
final_result = llm_with_tools.invoke(messages + [response] + [tool_result])
#                                    Human       AI(tool_call) +  Tool
final_result

AIMessage(content='3 + 5 = 8 입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 245, 'total_tokens': 257, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.2-2025-12-11', 'system_fingerprint': None, 'id': 'chatcmpl-Du81WPzQuSNwOZ1map8KdubCFHT8U', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ef78c-36c4-7660-9749-d50159a39e09-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 245, 'output_tokens': 12, 'total_tokens': 257, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

### 5-4. Tool Calling 전체 흐름

Tool Calling은 다음과 같은 메시지 순서로 동작합니다:

```
HumanMessage (사용자 질문)
    → AIMessage + tool_calls (LLM이 도구 호출 결정)
        → ToolMessage (도구 실행 결과)
            → AIMessage (최종 응답)
```

LLM이 여러 도구를 동시에 호출할 수도 있습니다 (Parallel Tool Calling).

In [58]:
from langchain.messages import HumanMessage

# Step 1: 사용자 질문
messages = [HumanMessage(content='오늘 날짜를 알려주고, 7 곱하기 8도 계산해줘.')]

# Step 2: LLM이 Tool Call 생성
ai_msg = llm_with_tools.invoke(messages)
messages.append(ai_msg)

print(f'LLM이 요청한 Tool Calls: {len(ai_msg.tool_calls)}개')
for tc in ai_msg.tool_calls:
    print(f'  -> {tc["name"]}({tc["args"]})')

LLM이 요청한 Tool Calls: 2개
  -> get_current_date({})
  -> multiply({'a': 7, 'b': 8})


In [59]:
# Step 3: 각 Tool Call 실행 후 메시지에 추가
for tc in ai_msg.tool_calls:
    result = tools_map[tc['name']].invoke(tc)
    messages.append(result)
    print(f'{tc["name"]} 실행 결과: {result.text}')

# Step 4: 최종 응답 생성
final = llm_with_tools.invoke(messages)
print(f'\n최종 응답: {final.text}')

get_current_date 실행 결과: 2026-06-24
multiply 실행 결과: 56

최종 응답: 오늘 날짜: 2026-06-24  
7 × 8 = 56


### 5-5. 메시지 흐름 시각화

위 과정의 전체 메시지 목록을 확인해봅시다.

In [60]:
messages.append(final)

for i, msg in enumerate(messages):
    msg_type = type(msg).__name__
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        calls = ', '.join([f"{tc['name']}({tc['args']})" for tc in msg.tool_calls])
        print(f'[{i}] {msg_type}: tool_calls=[{calls}]')
    elif hasattr(msg, 'tool_call_id'):
        print(f'[{i}] {msg_type}: ({msg.name}) -> {msg.text}')
    else:
        content_preview = msg.text[:80] + '...' if len(msg.text) > 80 else msg.text
        print(f'[{i}] {msg_type}: {content_preview}')

[0] HumanMessage: 오늘 날짜를 알려주고, 7 곱하기 8도 계산해줘.
[1] AIMessage: tool_calls=[get_current_date({}), multiply({'a': 7, 'b': 8})]
[2] ToolMessage: (get_current_date) -> 2026-06-24
[3] ToolMessage: (multiply) -> 56
[4] AIMessage: 오늘 날짜: 2026-06-24  
7 × 8 = 56


### 5-6. while 구문을 이용한 도구 호출 루프

실제 작업은 한 번의 도구 호출로 끝나지 않는 경우가 많습니다.    
도구 실행 결과를 확인한 뒤에 LLM이 다음 도구를 또 호출할 수도 있습니다.

다단계의 Tool Calling은 반복문을 통해 구현해 보겠습니다.

LLM의 출력인 `AIMessage`에 tool call이 있으면 도구를 실행하고 Context에 추가합니다.   
이후 다시 LLM을 호출합니다.     

In [64]:
from langchain.messages import HumanMessage

messages = [HumanMessage('''오늘 날짜를 확인하고, 모든 자릿수를 더한 결과를 알려줘.
매번 도구를 사용하기 전에 중간 과정을 요약하고 다음 단계를 설명해. 더하기는 반드시 add 도구를 써''')]
print('HumanMessage:', messages[-1].text)

max_turns = 20  # 무한 루프 방지를 위한 상한
turn = 0

while turn < max_turns:
    turn += 1
    ai_msg = llm_with_tools.invoke(messages)
    print('---- LLM 입력 ----')
    messages.append(ai_msg)
    # AIMessage Context에 추가
    print('AIMessage:', messages[-1].text)
    
    # 호출할 도구가 없으면 루프 종료
    if not ai_msg.tool_calls:
        break

    print(f'[Tool Call] ', end='')
    for tc in ai_msg.tool_calls:
        print(f'{tc["name"]}({tc["args"]})', end=', ')
    print('\n')    
    for tc in ai_msg.tool_calls:
        result = tools_map[tc['name']].invoke(tc)
        messages.append(result)    
        print('ToolMessage:', result.text)
        # 각 도구를 실행하고 결과를 Context에 추가

HumanMessage: 오늘 날짜를 확인하고, 모든 자릿수를 더한 결과를 알려줘.
매번 도구를 사용하기 전에 중간 과정을 요약하고 다음 단계를 설명해. 더하기는 반드시 add 도구를 써
---- LLM 입력 ----
AIMessage: 중간 과정 요약: 먼저 도구로 오늘 날짜(YYYY-MM-DD)를 가져와 연·월·일의 모든 숫자 자릿수를 나열하겠습니다.  
다음 단계: `get_current_date` 도구로 날짜를 조회하겠습니다.
[Tool Call] get_current_date({}), 

ToolMessage: 2026-06-24
---- LLM 입력 ----
AIMessage: 중간 과정 요약: 오늘 날짜는 2026-06-24이며, 숫자 자릿수는 2,0,2,6,0,6,2,4 입니다. 이제 이 자릿수들을 모두 더해야 합니다.  
다음 단계: 더하기는 `add` 도구로만 수행해야 하므로, 2+0부터 순서대로 누적 합을 `add`로 계산하겠습니다.
[Tool Call] add({'a': 2, 'b': 0}), 

ToolMessage: 2
---- LLM 입력 ----
AIMessage: 
[Tool Call] add({'a': 2, 'b': 2}), 

ToolMessage: 4
---- LLM 입력 ----
AIMessage: 
[Tool Call] add({'a': 4, 'b': 6}), 

ToolMessage: 10
---- LLM 입력 ----
AIMessage: 
[Tool Call] add({'a': 10, 'b': 0}), 

ToolMessage: 10
---- LLM 입력 ----
AIMessage: 
[Tool Call] add({'a': 10, 'b': 6}), 

ToolMessage: 16
---- LLM 입력 ----
AIMessage: 
[Tool Call] add({'a': 16, 'b': 2}), 

ToolMessage: 18
---- LLM 입력 ----
AIMessage: 
[Tool Call] add({'a': 1

---
## 6. 새로운 도구 연결하기

다양한 함수를 툴 콜링의 문법으로 연결할 수 있습니다.

In [65]:
@tool
def sleep_recommendation(age: int) -> str:
    """나이를 입력받아 권장 수면 시간을 안내합니다.

    Args:
        age: 나이 (정수)
    """
    if age < 6:
        hours = "10-13시간"
    elif age < 13:
        hours = "9-11시간"
    elif age < 18:
        hours = "8-10시간"
    elif age < 65:
        hours = "7-9시간"
    else:
        hours = "7-8시간"

    return f"권장 수면 시간: {hours}"


# 도구 직접 실행
print(sleep_recommendation.invoke({'age': 30}))

권장 수면 시간: 7-9시간


In [66]:
# LLM에 연결하여 Tool Calling 전체 흐름 실행
my_tools = [sleep_recommendation, add, get_current_date]
llm_with_tools = gpt.bind_tools(my_tools)

msgs = [HumanMessage('제가 31세 남성인데요, 하루에 6시간 정도 자는데 충분할까요?')]
ai_response = llm_with_tools.invoke(msgs)
msgs.append(ai_response)

# 도구 실행
my_tools_map = {t.name: t for t in my_tools}
for tc in ai_response.tool_calls:
    result = my_tools_map[tc['name']].invoke(tc)
    msgs.append(result)
    print(f'도구 실행: {tc["name"]} -> {result.text}')

# 최종 응답
final = llm_with_tools.invoke(msgs)
print(f'\n최종 응답: {final.text}')

도구 실행: sleep_recommendation -> 권장 수면 시간: 7-9시간

최종 응답: 31세 성인이라면 일반적인 권장 수면 시간은 **하루 7–9시간**입니다. 그래서 **평균 6시간 수면은 대체로 “부족한 편”**인 경우가 많습니다.

다만 “충분한지”는 시간만으로 딱 잘라 말하기 어렵고, 아래에 해당하면 6시간은 현재 몸에 부족하다고 보는 게 안전합니다.

- 아침에 개운하지 않고 **알람 없으면 더 자고 싶다**
- 낮에 **졸림/집중력 저하**가 있다(회의, 운전 중 포함)
- 주말에 **평일보다 1–2시간 이상 더 잠**으로 보충한다
- 카페인에 많이 의존하거나, 기분 변화/식욕 증가가 있다

반대로, **매일 거의 일정한 시간에 6시간 전후로 자도** 낮에 졸리지 않고 컨디션·집중력이 안정적이며 주말에도 크게 “몰아자기”가 없다면 개인차로 그럭저럭 맞는 경우도 있습니다. 하지만 장기적으로는 **7시간 이상으로 올려보는 걸 권장**합니다(예: 15–30분씩 서서히 늘리기).

원하시면  
1) 평일/주말 수면 시간표, 2) 낮 졸림 정도, 3) 카페인/운동/야식 여부를 알려주시면 6시간이 “괜찮은 편인지” 더 구체적으로 같이 판단해드릴게요.


---
## [실습] 커스텀 도구를 구성하여 Tool Calling 해보기

아래 조건에 맞는 커스텀 도구를 만들고, Tool Calling 전체 흐름을 실행해보세요.

요구사항:
1. 도구 1개 이상 새로 정의 (예: 환율 계산기, 온도 변환기, 텍스트 요약 등)
2. LLM에 `bind_tools`로 연결
3. Tool Calling 전체 흐름 실행 (HumanMessage - AIMessage - ToolMessage - 최종 응답)

도구의 docstring을 활용하여, LLM에게 해당 도구의 사용법을 전달하세요.

In [69]:
# tool 문법으로 도구 정의
# bind_tools로 도구 연결
# invoke로 실행

import random
from langchain_core.tools import tool

@tool
def evaluate_impulse_buy(item_name: str, price_krw: int) -> str:
    """충동구매하려는 물건과 가격을 분석하여, 직관적인 대안 가치 비교와 함께 구매 충동을 늦출 수 있는 무작위 미션을 제안합니다.

    Args:
        item_name: 구매하려는 물건의 이름 (예: '기계식 키보드', '스마트 워치')
        price_krw: 물건의 가격 (원 단위, 정수)
    """
    if price_krw <= 0:
        return "가격이 0원 이하입니다. 무료라면 부담 없이 선택하셔도 좋습니다."

    # 대안 화폐 단위 정의 (단위명, 기준 가격)
    alternatives = [
        ("뜨끈한 뼈다귀 해장국", 10000),
        ("아이스 아메리카노", 4500),
        ("신라면 봉지면", 1000),
        ("넷플릭스 광고형 스탠다드 1개월 구독", 5500),
        ("최저임금 기준 노동 시간 (시간)", 9860)
    ]
    
    # 무작위로 비교 대상 선택
    selected_alt, unit_price = random.choice(alternatives)
    ratio = price_krw / unit_price
    
    # 구매 결정을 유예하기 위한 무작위 미션
    missions = [
        "장바구니에만 넣어두고 정확히 24시간 뒤에 다시 결제창을 열어보세요. 그때도 갖고 싶다면 필요한 물건일 확률이 높습니다.",
        "이 물건이 가진 치명적인 단점 혹은 사용 빈도가 낮을 이유를 메모장에 3가지만 적어보세요.",
        "즉시 물을 한 컵 마시고 가벼운 스트레칭을 1분간 하세요. 일시적인 쇼핑 도파민 분비를 줄이는 데 도움이 됩니다.",
        "집에 이 물건과 비슷하거나 대체할 수 있는 물건이 이미 있는지 방 안을 한 바퀴 둘러보며 찾아보세요."
    ]
    selected_mission = random.choice(missions)
    
    # 결과 포맷팅
    result = (
        f"💸 [{item_name}] 소비 제어 분석 결과\n"
        f"- 입력 가격: {price_krw:,}원\n"
        f"- 체감 물가 비교: 이 금액은 '{selected_alt}' 약 {ratio:.1f}개(시간)에 해당하는 가치입니다.\n"
        f"- 추천 미션: {selected_mission}"
    )
    return result


# 도구 실행 테스트
print(evaluate_impulse_buy.invoke({'item_name': '노이즈 캔슬링 헤드폰', 'price_krw': 280000}))

💸 [노이즈 캔슬링 헤드폰] 소비 제어 분석 결과
- 입력 가격: 280,000원
- 체감 물가 비교: 이 금액은 '최저임금 기준 노동 시간 (시간)' 약 28.4개(시간)에 해당하는 가치입니다.
- 추천 미션: 즉시 물을 한 컵 마시고 가벼운 스트레칭을 1분간 하세요. 일시적인 쇼핑 도파민 분비를 줄이는 데 도움이 됩니다.


In [70]:
# LLM에 연결하여 Tool Calling 전체 흐름 실행
my_tools = [evaluate_impulse_buy, add, get_current_date]
llm_with_tools = gpt.bind_tools(my_tools)

msgs = [HumanMessage('그래픽카드(3000만원) 새로 사고 싶어요. 충동을 막아주세요.')]
ai_response = llm_with_tools.invoke(msgs)
msgs.append(ai_response)

# 도구 실행
my_tools_map = {t.name: t for t in my_tools}
for tc in ai_response.tool_calls:
    result = my_tools_map[tc['name']].invoke(tc)
    msgs.append(result)
    print(f'도구 실행: {tc["name"]} -> {result.text}')

# 최종 응답
final = llm_with_tools.invoke(msgs)
print(f'\n최종 응답: {final.text}')

도구 실행: evaluate_impulse_buy -> 💸 [그래픽카드] 소비 제어 분석 결과
- 입력 가격: 30,000,000원
- 체감 물가 비교: 이 금액은 '뜨끈한 뼈다귀 해장국' 약 3000.0개(시간)에 해당하는 가치입니다.
- 추천 미션: 장바구니에만 넣어두고 정확히 24시간 뒤에 다시 결제창을 열어보세요. 그때도 갖고 싶다면 필요한 물건일 확률이 높습니다.

최종 응답: 그래픽카드 3,000만원은 “업그레이드”라기보다 **프로젝트급 지출**이라서, 지금은 일단 결제 버튼에서 손 떼는 게 최선입니다. 아래대로만 해보세요.

## 1) 3,000만원의 대안 가치(현실 비교)
- 해장국(1만원 가정) 기준 **약 3,000그릇** 값입니다.
- PC 사용 목적이 게임/취미라면, 대부분의 경우 **체감 성능 이득 대비 지출 효율이 극도로 낮을 확률**이 큽니다. (특히 이미 상급기 보유 중이면 더더욱)

## 2) “구매 정당화 체크” 5문항 (3개 이상 NO면 보류)
1. 이 그래픽카드가 없어서 **지금 돈을 버는 작업이 막히는가?**
2. 지금 쓰는 GPU가 **업무/학업에서 시간을 실제로 손해** 보게 만드는가?
3. 필요한 VRAM/성능 요구치가 **명확한 소프트웨어/프로젝트로 증명**되는가?
4. 구매해도 **1년 내 중고 처분 시 손실을 감당**할 수 있는가?
5. 같은 목표를 **클라우드/렌탈/중고/한 단계 아래 모델**로 해결 못 하는가?

## 3) 충동을 늦추는 강제 장치(오늘 당장)
- **미션(24시간 룰)**: 장바구니까지만 넣고, **정확히 24시간 뒤**에 결제창을 다시 열기.  
  (지금 바로 결제는 금지)
- 그동안 할 일 2개:
  1) “왜 필요한지”를 한 줄로 쓰기: *“나는 ___ 때문에 필요하다.”*  
  2) “대안 3개” 적기: **1) 한 단계 아래 모델 2) 중고 3) 렌탈/클라우드**

## 4) 조건부 구매 규칙(허용되는 경우를 좁히기)
24시간 뒤에도 사고 싶다면, 아래 조건을 충족할 때만 진행:

---
## 정리

이번 실습에서 학습한 내용을 정리합니다.

- init_chat_model: 모델 이름만으로 다양한 LLM Provider를 통일된 인터페이스로 초기화
- Prompt Template: `ChatPromptTemplate`으로 재사용 가능한 시스템/사용자 프롬프트 구성
- 체인(Chain): `prompt | llm` 파이프 연산자로 프롬프트와 LLM을 연결하는 간결한 문법
- Structured Output: `with_structured_output()`으로 Pydantic 모델 기반 구조화된 데이터 생성
- @tool 데코레이터: 파이썬 함수를 LLM이 사용할 수 있는 도구로 변환. 이름과 docstring이 중요
- bind_tools: LLM에 도구 정보를 JSON Schema로 전달. LLM은 이를 바탕으로 도구 선택 및 인자 생성
- Tool Calling 흐름: `HumanMessage → AIMessage(tool_calls) → ToolMessage → AIMessage(최종 응답)`, 구조화 출력의 특별한 사례